# LLM Classifications analysis

In [1]:
import pandas as pd
from typing import Dict, Any
import re
import pandas as pd
import numpy as np

In [2]:
def enrich_program_metadata(df: pd.DataFrame, drop_filename: bool = False) -> pd.DataFrame:
    """
    Parse the 'filename' field and add:
        - program: canonical program family (10 values)
        - opt_level: O0/O1/O2/O3/Os/Ofast/default
        - compiler: gcc or clang

    Parameters
    ----------
    df : DataFrame
        Input dataframe containing column 'filename'.
    drop_filename : bool
        If True, removes the original 'filename' column.

    Returns
    -------
    DataFrame
        Enriched DataFrame with new columns.
    """

    # --------------------------------------------
    # 1) Canonical Program Family Mapping
    # --------------------------------------------
    program_map = {
        "picohttpparser":          "PicoHTTPParser",
        "cjson":                   "cJSON",
        "http_parser":             "Benoitc_HTTP",
        "parse_xml":               "CParserXML",
        "example":                 "CSimpleJSONParser",
        "network_analyzer":        "Network_Packet_Analyzer",
        "elf-parser":              "ELF_Parser",
        "pcap_parser":             "PCAP_Parser",
        "packcc":                  "Packcc",
        "calc":                    "YACC_Calculator",   # calc is also from YACC grammar
    }

    # --------------------------------------------
    # 2) Function to extract canonical program name
    # --------------------------------------------
    def extract_program(fname):
        for prefix, canonical in program_map.items():
            if fname.startswith(prefix):
                return canonical
        return "UNKNOWN"

    # --------------------------------------------
    # 3) Extract optimization level
    # --------------------------------------------
    def extract_opt_level(fname):
        # Order matters: check Ofast before O3, etc
        if "_Ofast" in fname:
            return "Ofast"
        if "_O3" in fname:
            return "O3"
        if "_O2" in fname:
            return "O2"
        if "_O1" in fname:
            return "O1"
        if "_O0" in fname:
            return "O0"
        if "_Os" in fname:
            return "Os"
        return "default"

    # --------------------------------------------
    # 4) Extract compiler
    # --------------------------------------------
    def extract_compiler(fname):
        return "clang" if "_clang" in fname else "gcc"

    # ------------------------------------------------
    # Apply all parsing functions
    # ------------------------------------------------
    df = df.copy()
    df["program"] = df["filename"].apply(extract_program)
    df["opt_level"] = df["filename"].apply(extract_opt_level)
    df["compiler"] = df["filename"].apply(extract_compiler)

    # Remove original filename if requested
    if drop_filename:
        df = df.drop(columns=["filename"])

    return df

In [3]:
def compute_program_recall(df: pd.DataFrame, program_name: str, code_column: str) -> Dict[str, Any]:
    """
    Compute recall and additional statistics for a specific *program* (not filename).

    Parameters
    ----------
    df : pandas.DataFrame
        The entire prediction dataset loaded from the CSV.
    program_name : str
        Canonical name of the program (matches column 'program').

    Returns
    -------
    Dict[str, Any]
        Dictionary with recall and detailed statistics.
    """

    # Filter rows belonging to this program
    prog_df = df[df["program"] == program_name]

    if prog_df.empty:
        raise ValueError(f"No entries found for program '{program_name}'")

    # ----------------------------------------------------------
    # Convert predicted & true labels to numeric, coercing errors
    # non-numeric labels become NaN and will be excluded automatically
    # ----------------------------------------------------------
    true = pd.to_numeric(prog_df["true_label"], errors="coerce")
    pred = pd.to_numeric(prog_df["predicted_label"], errors="coerce")

    # ----------------------------------------------------------
    # Missing code ("" or NaN)
    # ----------------------------------------------------------
    missing_code = ((prog_df[code_column].isna()) |
                      (prog_df[code_column] == "")).sum()

    # ----------------------------------------------------------
    # Recall computation only on valid numeric entries
    # ----------------------------------------------------------
    valid_mask = (~true.isna()) & (~pred.isna())
    true_valid = true[valid_mask]
    pred_valid = pred[valid_mask]

    tp = ((true_valid == 1) & (pred_valid == 1)).sum()
    actual_pos = (true_valid == 1).sum()

    if actual_pos == 0:
        recall = 1.0
    else:
        recall = tp / actual_pos

    # ----------------------------------------------------------
    # Predicted label statistics
    # ----------------------------------------------------------
    num_pred_0 = (pred_valid == 0).sum()
    num_pred_1 = (pred_valid == 1).sum()
    num_pred_unknown = pred.isna().sum()

    # ----------------------------------------------------------
    # Return statistics dictionary
    # ----------------------------------------------------------
    stats = {
        "program_name": program_name,
        "num_entries": len(prog_df),
        "num_valid_predictions": len(pred_valid),
        "num_missing_code": missing_code,

        # Prediction distribution
        "num_pred_0": num_pred_0,
        "num_pred_1": num_pred_1,
        "num_pred_unknown": num_pred_unknown,

        # Recall metrics
        "true_positives": tp,
        "actual_positives": actual_pos,
        "recall": recall,
    }

    return stats

## Mistral

### Decompiled Code LLM Classifications analysis

In [28]:
df = pd.read_csv("../Results/llm_classification_mistral/global_predictions_decompiled_code.csv")
# Shape of the dataframe
print(f"df.shape: {df.shape}")
# Display first few rows
df.head()

df.shape: (7898, 6)


,filename,function_name,address,decompiled_code,predicted_label,true_label
0,parse_xml,sym.deregister_tm_clones,0x401110,long long deregister_tm_clones()\n{\n if (t...,0,0
1,parse_xml,sym.register_tm_clones,0x401140,long long register_tm_clones()\n{\n if (tru...,0,0
2,parse_xml,sym.__do_global_dtors_aux,0x401180,extern char __bss_start;\n\nlong long __do_glo...,0,0
3,parse_xml,sym.frame_dummy,0x4011b0,long long frame_dummy()\n{\n return registe...,0,0
4,parse_xml,sym.skip_whitespace,0x4011cb,long long skip_whitespace(unsigned long a0)\n{...,0,1


In [29]:
# Remove the filename column and add: program, compiler, opt_level
df = enrich_program_metadata(df, drop_filename=True)

# Shape of the dataframe
print(f"df.shape: {df.shape}")
# Display first few rows
df.head()

df.shape: (7898, 8)


,function_name,address,decompiled_code,predicted_label,true_label,program,opt_level,compiler
0,sym.deregister_tm_clones,0x401110,long long deregister_tm_clones()\n{\n if (t...,0,0,CParserXML,default,gcc
1,sym.register_tm_clones,0x401140,long long register_tm_clones()\n{\n if (tru...,0,0,CParserXML,default,gcc
2,sym.__do_global_dtors_aux,0x401180,extern char __bss_start;\n\nlong long __do_glo...,0,0,CParserXML,default,gcc
3,sym.frame_dummy,0x4011b0,long long frame_dummy()\n{\n return registe...,0,0,CParserXML,default,gcc
4,sym.skip_whitespace,0x4011cb,long long skip_whitespace(unsigned long a0)\n{...,0,1,CParserXML,default,gcc


In [30]:
# List of unique programs
programs = df["program"].unique()
programs

array(['CParserXML', 'PicoHTTPParser', 'CSimpleJSONParser', 'cJSON',
       'Benoitc_HTTP', 'YACC_Calculator', 'Network_Packet_Analyzer',
       'ELF_Parser', 'PCAP_Parser', 'Packcc'], dtype=object)

In [31]:
recalls = []                    # Store recall statistics for each program
tot_missing_code = []           # Store total missing code statistics
tot_non_valid_predictions = []  # Store total non-valid predictions statistics

# Compute and print recall statistics for each program
for prog in programs:
    stats = compute_program_recall(df, prog, code_column="decompiled_code")
    print("-------------------------------------------------------------------------------------------------------------")
    print(f"Program: {prog}")
    print(f"  Number of entries: {stats['num_entries']}")
    print(f"  Number of valid predictions: {stats['num_valid_predictions']}")
    print(f"  Number of missing code: {stats['num_missing_code']}")
    print(f"  Predicted label distribution: 0s={stats['num_pred_0']}, 1s={stats['num_pred_1']}, unknown={stats['num_pred_unknown']}")
    print(f"  True Positives: {stats['true_positives']}")
    print(f"  Actual Positives: {stats['actual_positives']}")
    print(f"  Recall: {stats['recall']:.4f}")
    # Accumulate for overall statistics
    recalls.append(stats['recall'])
    tot_missing_code.append(stats['num_missing_code'])
    tot_non_valid_predictions.append(stats['num_entries'] - stats['num_valid_predictions'])

print(f"\n")
print("=============================================================================================================")
overall_recall_mean = np.mean(recalls)
overall_recall_std = np.std(recalls)
print(f"Overall Recall across all programs: {overall_recall_mean:.4f} ± {overall_recall_std:.4f}")
total_missing_code = sum(tot_missing_code)
total_non_valid_predictions = sum(tot_non_valid_predictions)
print(f"Total missing code entries across all programs: {total_missing_code}")
print(f"Total non-valid prediction entries across all programs: {total_non_valid_predictions}")
print(f"Total entries analyzed: {df.shape[0]}")

-------------------------------------------------------------------------------------------------------------
Program: CParserXML
  Number of entries: 362
  Number of valid predictions: 315
  Number of missing code: 45
  Predicted label distribution: 0s=158, 1s=157, unknown=47
  True Positives: 142
  Actual Positives: 185
  Recall: 0.7676
-------------------------------------------------------------------------------------------------------------
Program: PicoHTTPParser
  Number of entries: 760
  Number of valid predictions: 699
  Number of missing code: 58
  Predicted label distribution: 0s=286, 1s=413, unknown=61
  True Positives: 264
  Actual Positives: 323
  Recall: 0.8173
-------------------------------------------------------------------------------------------------------------
Program: CSimpleJSONParser
  Number of entries: 653
  Number of valid predictions: 604
  Number of missing code: 49
  Predicted label distribution: 0s=297, 1s=307, unknown=49
  True Positives: 146
  Actua

### VEX IR LLM Classifications analysis

In [32]:
df = pd.read_csv("../Results/llm_classification_mistral/global_predictions_vex_ir_code.csv")
# Shape of the dataframe
print(f"df.shape: {df.shape}")
# Display first few rows
df.head()

df.shape: (7898, 6)


,filename,function_name,address,vex_ir_code,predicted_label,true_label
0,parse_xml,sym.deregister_tm_clones,0x401110,"------ IMark(0x401110, 5, 0) ------\nPUT(offse...",1,0
1,parse_xml,sym.register_tm_clones,0x401140,"------ IMark(0x401140, 5, 0) ------\n------ IM...",1,0
2,parse_xml,sym.__do_global_dtors_aux,0x401180,"------ IMark(0x401180, 4, 0) ------\nPUT(offse...",1,0
3,parse_xml,sym.frame_dummy,0x4011b0,"------ IMark(0x4011b0, 4, 0) ------\n------ IM...",1,0
4,parse_xml,sym.skip_whitespace,0x4011cb,"------ IMark(0x40120a, 5, 0) ------\nt5 = GET:...",1,1


In [33]:
# Remove the filename column and add: program, compiler, opt_level
df = enrich_program_metadata(df, drop_filename=True)

# Shape of the dataframe
print(f"df.shape: {df.shape}")
# Display first few rows
df.head()

df.shape: (7898, 8)


,function_name,address,vex_ir_code,predicted_label,true_label,program,opt_level,compiler
0,sym.deregister_tm_clones,0x401110,"------ IMark(0x401110, 5, 0) ------\nPUT(offse...",1,0,CParserXML,default,gcc
1,sym.register_tm_clones,0x401140,"------ IMark(0x401140, 5, 0) ------\n------ IM...",1,0,CParserXML,default,gcc
2,sym.__do_global_dtors_aux,0x401180,"------ IMark(0x401180, 4, 0) ------\nPUT(offse...",1,0,CParserXML,default,gcc
3,sym.frame_dummy,0x4011b0,"------ IMark(0x4011b0, 4, 0) ------\n------ IM...",1,0,CParserXML,default,gcc
4,sym.skip_whitespace,0x4011cb,"------ IMark(0x40120a, 5, 0) ------\nt5 = GET:...",1,1,CParserXML,default,gcc


In [34]:
# List of unique programs
programs = df["program"].unique()
programs

array(['CParserXML', 'PicoHTTPParser', 'CSimpleJSONParser', 'cJSON',
       'Benoitc_HTTP', 'YACC_Calculator', 'Network_Packet_Analyzer',
       'ELF_Parser', 'PCAP_Parser', 'Packcc'], dtype=object)

In [35]:
recalls = []                    # Store recall statistics for each program
tot_missing_code = []           # Store total missing code statistics
tot_non_valid_predictions = []  # Store total non-valid predictions statistics

# Compute and print recall statistics for each program
for prog in programs:
    stats = compute_program_recall(df, prog, code_column="vex_ir_code")
    print("-------------------------------------------------------------------------------------------------------------")
    print(f"Program: {prog}")
    print(f"  Number of entries: {stats['num_entries']}")
    print(f"  Number of valid predictions: {stats['num_valid_predictions']}")
    print(f"  Number of missing code: {stats['num_missing_code']}")
    print(f"  Predicted label distribution: 0s={stats['num_pred_0']}, 1s={stats['num_pred_1']}, unknown={stats['num_pred_unknown']}")
    print(f"  True Positives: {stats['true_positives']}")
    print(f"  Actual Positives: {stats['actual_positives']}")
    print(f"  Recall: {stats['recall']:.4f}")
    # Accumulate for overall statistics
    recalls.append(stats['recall'])
    tot_missing_code.append(stats['num_missing_code'])
    tot_non_valid_predictions.append(stats['num_entries'] - stats['num_valid_predictions'])

print(f"\n")
print("=============================================================================================================")
overall_recall_mean = np.mean(recalls)
overall_recall_std = np.std(recalls)
print(f"Overall Recall across all programs: {overall_recall_mean:.4f} ± {overall_recall_std:.4f}")
total_missing_code = sum(tot_missing_code)
total_non_valid_predictions = sum(tot_non_valid_predictions)
print(f"Total missing code entries across all programs: {total_missing_code}")
print(f"Total non-valid prediction entries across all programs: {total_non_valid_predictions}")
print(f"Total entries analyzed: {df.shape[0]}")

-------------------------------------------------------------------------------------------------------------
Program: CParserXML
  Number of entries: 362
  Number of valid predictions: 330
  Number of missing code: 0
  Predicted label distribution: 0s=69, 1s=261, unknown=32
  True Positives: 116
  Actual Positives: 173
  Recall: 0.6705
-------------------------------------------------------------------------------------------------------------
Program: PicoHTTPParser
  Number of entries: 760
  Number of valid predictions: 684
  Number of missing code: 0
  Predicted label distribution: 0s=289, 1s=395, unknown=76
  True Positives: 127
  Actual Positives: 295
  Recall: 0.4305
-------------------------------------------------------------------------------------------------------------
Program: CSimpleJSONParser
  Number of entries: 653
  Number of valid predictions: 589
  Number of missing code: 0
  Predicted label distribution: 0s=210, 1s=379, unknown=64
  True Positives: 65
  Actual Pos

### Assembly Code LLM Classifications analysis

In [42]:
df = pd.read_csv("../Results/llm_classification_mistral/global_predictions_assembly_code.csv")
# Shape of the dataframe
print(f"df.shape: {df.shape}")
# Display first few rows
df.head()

df.shape: (7898, 6)


,filename,function_name,address,assembly_code,predicted_label,true_label
0,parse_xml,sym.deregister_tm_clones,0x401110,"0x401110:\tmov\teax, 0x405058\n0x401115:\tcmp\...",0,0
1,parse_xml,sym.register_tm_clones,0x401140,"0x401140:\tmov\tesi, 0x405058\n0x401145:\tsub\...",1,0
2,parse_xml,sym.__do_global_dtors_aux,0x401180,0x401180:\tendbr64\t\n0x401184:\tcmp\tbyte ptr...,1,0
3,parse_xml,sym.frame_dummy,0x4011b0,0x4011b0:\tendbr64\t\n0x4011b4:\tjmp\t0x401140,0,0
4,parse_xml,sym.skip_whitespace,0x4011cb,"0x40120a:\tadd\tqword ptr [rbp - 8], 1\n0x4012...",1,1


In [43]:
# Remove the filename column and add: program, compiler, opt_level
df = enrich_program_metadata(df, drop_filename=True)

# Shape of the dataframe
print(f"df.shape: {df.shape}")
# Display first few rows
df.head()

df.shape: (7898, 8)


,function_name,address,assembly_code,predicted_label,true_label,program,opt_level,compiler
0,sym.deregister_tm_clones,0x401110,"0x401110:\tmov\teax, 0x405058\n0x401115:\tcmp\...",0,0,CParserXML,default,gcc
1,sym.register_tm_clones,0x401140,"0x401140:\tmov\tesi, 0x405058\n0x401145:\tsub\...",1,0,CParserXML,default,gcc
2,sym.__do_global_dtors_aux,0x401180,0x401180:\tendbr64\t\n0x401184:\tcmp\tbyte ptr...,1,0,CParserXML,default,gcc
3,sym.frame_dummy,0x4011b0,0x4011b0:\tendbr64\t\n0x4011b4:\tjmp\t0x401140,0,0,CParserXML,default,gcc
4,sym.skip_whitespace,0x4011cb,"0x40120a:\tadd\tqword ptr [rbp - 8], 1\n0x4012...",1,1,CParserXML,default,gcc


In [44]:
# List of unique programs
programs = df["program"].unique()
programs

array(['CParserXML', 'PicoHTTPParser', 'CSimpleJSONParser', 'cJSON',
       'Benoitc_HTTP', 'YACC_Calculator', 'Network_Packet_Analyzer',
       'ELF_Parser', 'PCAP_Parser', 'Packcc'], dtype=object)

In [45]:
recalls = []                    # Store recall statistics for each program
tot_missing_code = []           # Store total missing code statistics
tot_non_valid_predictions = []  # Store total non-valid predictions statistics

# Compute and print recall statistics for each program
for prog in programs:
    stats = compute_program_recall(df, prog, code_column="assembly_code")
    print("-------------------------------------------------------------------------------------------------------------")
    print(f"Program: {prog}")
    print(f"  Number of entries: {stats['num_entries']}")
    print(f"  Number of valid predictions: {stats['num_valid_predictions']}")
    print(f"  Number of missing code: {stats['num_missing_code']}")
    print(f"  Predicted label distribution: 0s={stats['num_pred_0']}, 1s={stats['num_pred_1']}, unknown={stats['num_pred_unknown']}")
    print(f"  True Positives: {stats['true_positives']}")
    print(f"  Actual Positives: {stats['actual_positives']}")
    print(f"  Recall: {stats['recall']:.4f}")
    # Accumulate for overall statistics
    recalls.append(stats['recall'])
    tot_missing_code.append(stats['num_missing_code'])
    tot_non_valid_predictions.append(stats['num_entries'] - stats['num_valid_predictions'])

print(f"\n")
print("=============================================================================================================")
overall_recall_mean = np.mean(recalls)
overall_recall_std = np.std(recalls)
print(f"Overall Recall across all programs: {overall_recall_mean:.4f} ± {overall_recall_std:.4f}")
total_missing_code = sum(tot_missing_code)
total_non_valid_predictions = sum(tot_non_valid_predictions)
print(f"Total missing code entries across all programs: {total_missing_code}")
print(f"Total non-valid prediction entries across all programs: {total_non_valid_predictions}")
print(f"Total entries analyzed: {df.shape[0]}")

-------------------------------------------------------------------------------------------------------------
Program: CParserXML
  Number of entries: 362
  Number of valid predictions: 351
  Number of missing code: 0
  Predicted label distribution: 0s=92, 1s=259, unknown=11
  True Positives: 148
  Actual Positives: 193
  Recall: 0.7668
-------------------------------------------------------------------------------------------------------------
Program: PicoHTTPParser
  Number of entries: 760
  Number of valid predictions: 735
  Number of missing code: 0
  Predicted label distribution: 0s=133, 1s=602, unknown=25
  True Positives: 226
  Actual Positives: 310
  Recall: 0.7290
-------------------------------------------------------------------------------------------------------------
Program: CSimpleJSONParser
  Number of entries: 653
  Number of valid predictions: 647
  Number of missing code: 0
  Predicted label distribution: 0s=74, 1s=573, unknown=6
  True Positives: 168
  Actual Posi

## Codellama:34b

### Decompiled Code LLM Classifications analysis

In [4]:
df = pd.read_csv("../Results/llm_classification_codellama_34b/global_predictions_decompiled_code.csv")
# Shape of the dataframe
print(f"df.shape: {df.shape}")
# Display first few rows
df.head()

df.shape: (7898, 6)


,filename,function_name,address,decompiled_code,predicted_label,true_label
0,parse_xml,sym.deregister_tm_clones,0x401110,long long deregister_tm_clones()\n{\n if (t...,0,0
1,parse_xml,sym.register_tm_clones,0x401140,long long register_tm_clones()\n{\n if (tru...,1,0
2,parse_xml,sym.__do_global_dtors_aux,0x401180,extern char __bss_start;\n\nlong long __do_glo...,0,0
3,parse_xml,sym.frame_dummy,0x4011b0,long long frame_dummy()\n{\n return registe...,0,0
4,parse_xml,sym.skip_whitespace,0x4011cb,long long skip_whitespace(unsigned long a0)\n{...,1,1


In [5]:
# Remove the filename column and add: program, compiler, opt_level
df = enrich_program_metadata(df, drop_filename=True)

# Shape of the dataframe
print(f"df.shape: {df.shape}")
# Display first few rows
df.head()

df.shape: (7898, 8)


,function_name,address,decompiled_code,predicted_label,true_label,program,opt_level,compiler
0,sym.deregister_tm_clones,0x401110,long long deregister_tm_clones()\n{\n if (t...,0,0,CParserXML,default,gcc
1,sym.register_tm_clones,0x401140,long long register_tm_clones()\n{\n if (tru...,1,0,CParserXML,default,gcc
2,sym.__do_global_dtors_aux,0x401180,extern char __bss_start;\n\nlong long __do_glo...,0,0,CParserXML,default,gcc
3,sym.frame_dummy,0x4011b0,long long frame_dummy()\n{\n return registe...,0,0,CParserXML,default,gcc
4,sym.skip_whitespace,0x4011cb,long long skip_whitespace(unsigned long a0)\n{...,1,1,CParserXML,default,gcc


In [6]:
# List of unique programs
programs = df["program"].unique()
programs

array(['CParserXML', 'PicoHTTPParser', 'CSimpleJSONParser', 'cJSON',
       'Benoitc_HTTP', 'YACC_Calculator', 'Network_Packet_Analyzer',
       'ELF_Parser', 'PCAP_Parser', 'Packcc'], dtype=object)

In [7]:
recalls = []                    # Store recall statistics for each program
tot_missing_code = []           # Store total missing code statistics
tot_non_valid_predictions = []  # Store total non-valid predictions statistics

# Compute and print recall statistics for each program
for prog in programs:
    stats = compute_program_recall(df, prog, code_column="decompiled_code")
    print("-------------------------------------------------------------------------------------------------------------")
    print(f"Program: {prog}")
    print(f"  Number of entries: {stats['num_entries']}")
    print(f"  Number of valid predictions: {stats['num_valid_predictions']}")
    print(f"  Number of missing code: {stats['num_missing_code']}")
    print(f"  Predicted label distribution: 0s={stats['num_pred_0']}, 1s={stats['num_pred_1']}, unknown={stats['num_pred_unknown']}")
    print(f"  True Positives: {stats['true_positives']}")
    print(f"  Actual Positives: {stats['actual_positives']}")
    print(f"  Recall: {stats['recall']:.4f}")
    # Accumulate for overall statistics
    recalls.append(stats['recall'])
    tot_missing_code.append(stats['num_missing_code'])
    tot_non_valid_predictions.append(stats['num_entries'] - stats['num_valid_predictions'])

print(f"\n")
print("=============================================================================================================")
overall_recall_mean = np.mean(recalls)
overall_recall_std = np.std(recalls)
print(f"Overall Recall across all programs: {overall_recall_mean:.4f} ± {overall_recall_std:.4f}")
total_missing_code = sum(tot_missing_code)
total_non_valid_predictions = sum(tot_non_valid_predictions)
print(f"Total missing code entries across all programs: {total_missing_code}")
print(f"Total non-valid prediction entries across all programs: {total_non_valid_predictions}")
print(f"Total entries analyzed: {df.shape[0]}")

-------------------------------------------------------------------------------------------------------------
Program: CParserXML
  Number of entries: 362
  Number of valid predictions: 288
  Number of missing code: 45
  Predicted label distribution: 0s=147, 1s=141, unknown=74
  True Positives: 90
  Actual Positives: 160
  Recall: 0.5625
-------------------------------------------------------------------------------------------------------------
Program: PicoHTTPParser
  Number of entries: 760
  Number of valid predictions: 600
  Number of missing code: 59
  Predicted label distribution: 0s=251, 1s=349, unknown=160
  True Positives: 174
  Actual Positives: 261
  Recall: 0.6667
-------------------------------------------------------------------------------------------------------------
Program: CSimpleJSONParser
  Number of entries: 653
  Number of valid predictions: 548
  Number of missing code: 49
  Predicted label distribution: 0s=247, 1s=301, unknown=105
  True Positives: 99
  Actua

### VEX IR LLM Classifications analysis

In [4]:
df = pd.read_csv("../Results/llm_classification_codellama_34b/global_predictions_vex_ir_code.csv")
# Shape of the dataframe
print(f"df.shape: {df.shape}")
# Display first few rows
df.head()

df.shape: (7898, 6)


,filename,function_name,address,vex_ir_code,predicted_label,true_label
0,parse_xml,sym.deregister_tm_clones,0x401110,"------ IMark(0x401110, 5, 0) ------\nPUT(offse...",1,0
1,parse_xml,sym.register_tm_clones,0x401140,"------ IMark(0x401140, 5, 0) ------\n------ IM...",0,0
2,parse_xml,sym.__do_global_dtors_aux,0x401180,"------ IMark(0x401180, 4, 0) ------\nPUT(offse...",unknown,0
3,parse_xml,sym.frame_dummy,0x4011b0,"------ IMark(0x4011b0, 4, 0) ------\n------ IM...",0,0
4,parse_xml,sym.skip_whitespace,0x4011cb,"------ IMark(0x40120a, 5, 0) ------\nt5 = GET:...",unknown,1


In [5]:
# Remove the filename column and add: program, compiler, opt_level
df = enrich_program_metadata(df, drop_filename=True)

# Shape of the dataframe
print(f"df.shape: {df.shape}")
# Display first few rows
df.head()

df.shape: (7898, 8)


,function_name,address,vex_ir_code,predicted_label,true_label,program,opt_level,compiler
0,sym.deregister_tm_clones,0x401110,"------ IMark(0x401110, 5, 0) ------\nPUT(offse...",1,0,CParserXML,default,gcc
1,sym.register_tm_clones,0x401140,"------ IMark(0x401140, 5, 0) ------\n------ IM...",0,0,CParserXML,default,gcc
2,sym.__do_global_dtors_aux,0x401180,"------ IMark(0x401180, 4, 0) ------\nPUT(offse...",unknown,0,CParserXML,default,gcc
3,sym.frame_dummy,0x4011b0,"------ IMark(0x4011b0, 4, 0) ------\n------ IM...",0,0,CParserXML,default,gcc
4,sym.skip_whitespace,0x4011cb,"------ IMark(0x40120a, 5, 0) ------\nt5 = GET:...",unknown,1,CParserXML,default,gcc


In [6]:
# List of unique programs
programs = df["program"].unique()
programs

array(['CParserXML', 'PicoHTTPParser', 'CSimpleJSONParser', 'cJSON',
       'Benoitc_HTTP', 'YACC_Calculator', 'Network_Packet_Analyzer',
       'ELF_Parser', 'PCAP_Parser', 'Packcc'], dtype=object)

In [7]:
recalls = []                    # Store recall statistics for each program
tot_missing_code = []           # Store total missing code statistics
tot_non_valid_predictions = []  # Store total non-valid predictions statistics

# Compute and print recall statistics for each program
for prog in programs:
    stats = compute_program_recall(df, prog, code_column="vex_ir_code")
    print("-------------------------------------------------------------------------------------------------------------")
    print(f"Program: {prog}")
    print(f"  Number of entries: {stats['num_entries']}")
    print(f"  Number of valid predictions: {stats['num_valid_predictions']}")
    print(f"  Number of missing code: {stats['num_missing_code']}")
    print(f"  Predicted label distribution: 0s={stats['num_pred_0']}, 1s={stats['num_pred_1']}, unknown={stats['num_pred_unknown']}")
    print(f"  True Positives: {stats['true_positives']}")
    print(f"  Actual Positives: {stats['actual_positives']}")
    print(f"  Recall: {stats['recall']:.4f}")
    # Accumulate for overall statistics
    recalls.append(stats['recall'])
    tot_missing_code.append(stats['num_missing_code'])
    tot_non_valid_predictions.append(stats['num_entries'] - stats['num_valid_predictions'])

print(f"\n")
print("=============================================================================================================")
overall_recall_mean = np.mean(recalls)
overall_recall_std = np.std(recalls)
print(f"Overall Recall across all programs: {overall_recall_mean:.4f} ± {overall_recall_std:.4f}")
total_missing_code = sum(tot_missing_code)
total_non_valid_predictions = sum(tot_non_valid_predictions)
print(f"Total missing code entries across all programs: {total_missing_code}")
print(f"Total non-valid prediction entries across all programs: {total_non_valid_predictions}")
print(f"Total entries analyzed: {df.shape[0]}")

-------------------------------------------------------------------------------------------------------------
Program: CParserXML
  Number of entries: 362
  Number of valid predictions: 234
  Number of missing code: 0
  Predicted label distribution: 0s=102, 1s=132, unknown=128
  True Positives: 64
  Actual Positives: 116
  Recall: 0.5517
-------------------------------------------------------------------------------------------------------------
Program: PicoHTTPParser
  Number of entries: 760
  Number of valid predictions: 439
  Number of missing code: 0
  Predicted label distribution: 0s=199, 1s=240, unknown=321
  True Positives: 82
  Actual Positives: 164
  Recall: 0.5000
-------------------------------------------------------------------------------------------------------------
Program: CSimpleJSONParser
  Number of entries: 653
  Number of valid predictions: 423
  Number of missing code: 0
  Predicted label distribution: 0s=196, 1s=227, unknown=230
  True Positives: 51
  Actual P

### Assembly Code LLM Classifications analysis

In [46]:
df = pd.read_csv("../Results/llm_classification_codellama_34b/global_predictions_assembly_code.csv")
# Shape of the dataframe
print(f"df.shape: {df.shape}")
# Display first few rows
df.head()

df.shape: (7898, 6)


,filename,function_name,address,assembly_code,predicted_label,true_label
0,parse_xml,sym.deregister_tm_clones,0x401110,"0x401110:\tmov\teax, 0x405058\n0x401115:\tcmp\...",0,0
1,parse_xml,sym.register_tm_clones,0x401140,"0x401140:\tmov\tesi, 0x405058\n0x401145:\tsub\...",0,0
2,parse_xml,sym.__do_global_dtors_aux,0x401180,0x401180:\tendbr64\t\n0x401184:\tcmp\tbyte ptr...,0,0
3,parse_xml,sym.frame_dummy,0x4011b0,0x4011b0:\tendbr64\t\n0x4011b4:\tjmp\t0x401140,0,0
4,parse_xml,sym.skip_whitespace,0x4011cb,"0x40120a:\tadd\tqword ptr [rbp - 8], 1\n0x4012...",0,1


In [47]:
# Remove the filename column and add: program, compiler, opt_level
df = enrich_program_metadata(df, drop_filename=True)

# Shape of the dataframe
print(f"df.shape: {df.shape}")
# Display first few rows
df.head()

df.shape: (7898, 8)


,function_name,address,assembly_code,predicted_label,true_label,program,opt_level,compiler
0,sym.deregister_tm_clones,0x401110,"0x401110:\tmov\teax, 0x405058\n0x401115:\tcmp\...",0,0,CParserXML,default,gcc
1,sym.register_tm_clones,0x401140,"0x401140:\tmov\tesi, 0x405058\n0x401145:\tsub\...",0,0,CParserXML,default,gcc
2,sym.__do_global_dtors_aux,0x401180,0x401180:\tendbr64\t\n0x401184:\tcmp\tbyte ptr...,0,0,CParserXML,default,gcc
3,sym.frame_dummy,0x4011b0,0x4011b0:\tendbr64\t\n0x4011b4:\tjmp\t0x401140,0,0,CParserXML,default,gcc
4,sym.skip_whitespace,0x4011cb,"0x40120a:\tadd\tqword ptr [rbp - 8], 1\n0x4012...",0,1,CParserXML,default,gcc


In [48]:
# List of unique programs
programs = df["program"].unique()
programs

array(['CParserXML', 'PicoHTTPParser', 'CSimpleJSONParser', 'cJSON',
       'Benoitc_HTTP', 'YACC_Calculator', 'Network_Packet_Analyzer',
       'ELF_Parser', 'PCAP_Parser', 'Packcc'], dtype=object)

In [49]:
recalls = []                    # Store recall statistics for each program
tot_missing_code = []           # Store total missing code statistics
tot_non_valid_predictions = []  # Store total non-valid predictions statistics

# Compute and print recall statistics for each program
for prog in programs:
    stats = compute_program_recall(df, prog, code_column="assembly_code")
    print("-------------------------------------------------------------------------------------------------------------")
    print(f"Program: {prog}")
    print(f"  Number of entries: {stats['num_entries']}")
    print(f"  Number of valid predictions: {stats['num_valid_predictions']}")
    print(f"  Number of missing code: {stats['num_missing_code']}")
    print(f"  Predicted label distribution: 0s={stats['num_pred_0']}, 1s={stats['num_pred_1']}, unknown={stats['num_pred_unknown']}")
    print(f"  True Positives: {stats['true_positives']}")
    print(f"  Actual Positives: {stats['actual_positives']}")
    print(f"  Recall: {stats['recall']:.4f}")
    # Accumulate for overall statistics
    recalls.append(stats['recall'])
    tot_missing_code.append(stats['num_missing_code'])
    tot_non_valid_predictions.append(stats['num_entries'] - stats['num_valid_predictions'])

print(f"\n")
print("=============================================================================================================")
overall_recall_mean = np.mean(recalls)
overall_recall_std = np.std(recalls)
print(f"Overall Recall across all programs: {overall_recall_mean:.4f} ± {overall_recall_std:.4f}")
total_missing_code = sum(tot_missing_code)
total_non_valid_predictions = sum(tot_non_valid_predictions)
print(f"Total missing code entries across all programs: {total_missing_code}")
print(f"Total non-valid prediction entries across all programs: {total_non_valid_predictions}")
print(f"Total entries analyzed: {df.shape[0]}")

-------------------------------------------------------------------------------------------------------------
Program: CParserXML
  Number of entries: 362
  Number of valid predictions: 283
  Number of missing code: 0
  Predicted label distribution: 0s=150, 1s=133, unknown=79
  True Positives: 75
  Actual Positives: 135
  Recall: 0.5556
-------------------------------------------------------------------------------------------------------------
Program: PicoHTTPParser
  Number of entries: 760
  Number of valid predictions: 562
  Number of missing code: 0
  Predicted label distribution: 0s=271, 1s=291, unknown=198
  True Positives: 111
  Actual Positives: 220
  Recall: 0.5045
-------------------------------------------------------------------------------------------------------------
Program: CSimpleJSONParser
  Number of entries: 653
  Number of valid predictions: 532
  Number of missing code: 0
  Predicted label distribution: 0s=218, 1s=314, unknown=121
  True Positives: 89
  Actual P

## Qwen3-Coder:30b

### Decompiled Code LLM Classifications analysis

In [8]:
df = pd.read_csv("../Results/llm_classification_qwen3_coder_30b/global_predictions_decompiled_code.csv")
# Shape of the dataframe
print(f"df.shape: {df.shape}")
# Display first few rows
df.head()

df.shape: (7898, 6)


,filename,function_name,address,decompiled_code,predicted_label,true_label
0,parse_xml,sym.deregister_tm_clones,0x401110,long long deregister_tm_clones()\n{\n if (t...,0,0
1,parse_xml,sym.register_tm_clones,0x401140,long long register_tm_clones()\n{\n if (tru...,0,0
2,parse_xml,sym.__do_global_dtors_aux,0x401180,extern char __bss_start;\n\nlong long __do_glo...,0,0
3,parse_xml,sym.frame_dummy,0x4011b0,long long frame_dummy()\n{\n return registe...,0,0
4,parse_xml,sym.skip_whitespace,0x4011cb,long long skip_whitespace(unsigned long a0)\n{...,0,1


In [9]:
# Remove the filename column and add: program, compiler, opt_level
df = enrich_program_metadata(df, drop_filename=True)

# Shape of the dataframe
print(f"df.shape: {df.shape}")
# Display first few rows
df.head()

df.shape: (7898, 8)


,function_name,address,decompiled_code,predicted_label,true_label,program,opt_level,compiler
0,sym.deregister_tm_clones,0x401110,long long deregister_tm_clones()\n{\n if (t...,0,0,CParserXML,default,gcc
1,sym.register_tm_clones,0x401140,long long register_tm_clones()\n{\n if (tru...,0,0,CParserXML,default,gcc
2,sym.__do_global_dtors_aux,0x401180,extern char __bss_start;\n\nlong long __do_glo...,0,0,CParserXML,default,gcc
3,sym.frame_dummy,0x4011b0,long long frame_dummy()\n{\n return registe...,0,0,CParserXML,default,gcc
4,sym.skip_whitespace,0x4011cb,long long skip_whitespace(unsigned long a0)\n{...,0,1,CParserXML,default,gcc


In [10]:
# List of unique programs
programs = df["program"].unique()
programs

array(['CParserXML', 'PicoHTTPParser', 'CSimpleJSONParser', 'cJSON',
       'Benoitc_HTTP', 'YACC_Calculator', 'Network_Packet_Analyzer',
       'ELF_Parser', 'PCAP_Parser', 'Packcc'], dtype=object)

In [11]:
recalls = []                    # Store recall statistics for each program
tot_missing_code = []           # Store total missing code statistics
tot_non_valid_predictions = []  # Store total non-valid predictions statistics

# Compute and print recall statistics for each program
for prog in programs:
    stats = compute_program_recall(df, prog, code_column="decompiled_code")
    print("-------------------------------------------------------------------------------------------------------------")
    print(f"Program: {prog}")
    print(f"  Number of entries: {stats['num_entries']}")
    print(f"  Number of valid predictions: {stats['num_valid_predictions']}")
    print(f"  Number of missing code: {stats['num_missing_code']}")
    print(f"  Predicted label distribution: 0s={stats['num_pred_0']}, 1s={stats['num_pred_1']}, unknown={stats['num_pred_unknown']}")
    print(f"  True Positives: {stats['true_positives']}")
    print(f"  Actual Positives: {stats['actual_positives']}")
    print(f"  Recall: {stats['recall']:.4f}")
    # Accumulate for overall statistics
    recalls.append(stats['recall'])
    tot_missing_code.append(stats['num_missing_code'])
    tot_non_valid_predictions.append(stats['num_entries'] - stats['num_valid_predictions'])

print(f"\n")
print("=============================================================================================================")
overall_recall_mean = np.mean(recalls)
overall_recall_std = np.std(recalls)
print(f"Overall Recall across all programs: {overall_recall_mean:.4f} ± {overall_recall_std:.4f}")
total_missing_code = sum(tot_missing_code)
total_non_valid_predictions = sum(tot_non_valid_predictions)
print(f"Total missing code entries across all programs: {total_missing_code}")
print(f"Total non-valid prediction entries across all programs: {total_non_valid_predictions}")
print(f"Total entries analyzed: {df.shape[0]}")

-------------------------------------------------------------------------------------------------------------
Program: CParserXML
  Number of entries: 362
  Number of valid predictions: 315
  Number of missing code: 45
  Predicted label distribution: 0s=224, 1s=91, unknown=47
  True Positives: 91
  Actual Positives: 185
  Recall: 0.4919
-------------------------------------------------------------------------------------------------------------
Program: PicoHTTPParser
  Number of entries: 760
  Number of valid predictions: 692
  Number of missing code: 58
  Predicted label distribution: 0s=378, 1s=314, unknown=68
  True Positives: 216
  Actual Positives: 316
  Recall: 0.6835
-------------------------------------------------------------------------------------------------------------
Program: CSimpleJSONParser
  Number of entries: 653
  Number of valid predictions: 604
  Number of missing code: 49
  Predicted label distribution: 0s=388, 1s=216, unknown=49
  True Positives: 133
  Actual 

### VEX IR LLM Classifications analysis

In [4]:
df = pd.read_csv("../Results/llm_classification_qwen3_coder_30b/global_predictions_vex_ir_code.csv")
# Shape of the dataframe
print(f"df.shape: {df.shape}")
# Display first few rows
df.head()

df.shape: (7898, 6)


,filename,function_name,address,vex_ir_code,predicted_label,true_label
0,parse_xml,sym.deregister_tm_clones,0x401110,"------ IMark(0x401110, 5, 0) ------\nPUT(offse...",1,0
1,parse_xml,sym.register_tm_clones,0x401140,"------ IMark(0x401140, 5, 0) ------\n------ IM...",1,0
2,parse_xml,sym.__do_global_dtors_aux,0x401180,"------ IMark(0x401180, 4, 0) ------\nPUT(offse...",1,0
3,parse_xml,sym.frame_dummy,0x4011b0,"------ IMark(0x4011b0, 4, 0) ------\n------ IM...",0,0
4,parse_xml,sym.skip_whitespace,0x4011cb,"------ IMark(0x40120a, 5, 0) ------\nt5 = GET:...",1,1


In [5]:
# Remove the filename column and add: program, compiler, opt_level
df = enrich_program_metadata(df, drop_filename=True)

# Shape of the dataframe
print(f"df.shape: {df.shape}")
# Display first few rows
df.head()

df.shape: (7898, 8)


,function_name,address,vex_ir_code,predicted_label,true_label,program,opt_level,compiler
0,sym.deregister_tm_clones,0x401110,"------ IMark(0x401110, 5, 0) ------\nPUT(offse...",1,0,CParserXML,default,gcc
1,sym.register_tm_clones,0x401140,"------ IMark(0x401140, 5, 0) ------\n------ IM...",1,0,CParserXML,default,gcc
2,sym.__do_global_dtors_aux,0x401180,"------ IMark(0x401180, 4, 0) ------\nPUT(offse...",1,0,CParserXML,default,gcc
3,sym.frame_dummy,0x4011b0,"------ IMark(0x4011b0, 4, 0) ------\n------ IM...",0,0,CParserXML,default,gcc
4,sym.skip_whitespace,0x4011cb,"------ IMark(0x40120a, 5, 0) ------\nt5 = GET:...",1,1,CParserXML,default,gcc


In [6]:
# List of unique programs
programs = df["program"].unique()
programs

array(['CParserXML', 'PicoHTTPParser', 'CSimpleJSONParser', 'cJSON',
       'Benoitc_HTTP', 'YACC_Calculator', 'Network_Packet_Analyzer',
       'ELF_Parser', 'PCAP_Parser', 'Packcc'], dtype=object)

In [7]:
recalls = []                    # Store recall statistics for each program
tot_missing_code = []           # Store total missing code statistics
tot_non_valid_predictions = []  # Store total non-valid predictions statistics

# Compute and print recall statistics for each program
for prog in programs:
    stats = compute_program_recall(df, prog, code_column="vex_ir_code")
    print("-------------------------------------------------------------------------------------------------------------")
    print(f"Program: {prog}")
    print(f"  Number of entries: {stats['num_entries']}")
    print(f"  Number of valid predictions: {stats['num_valid_predictions']}")
    print(f"  Number of missing code: {stats['num_missing_code']}")
    print(f"  Predicted label distribution: 0s={stats['num_pred_0']}, 1s={stats['num_pred_1']}, unknown={stats['num_pred_unknown']}")
    print(f"  True Positives: {stats['true_positives']}")
    print(f"  Actual Positives: {stats['actual_positives']}")
    print(f"  Recall: {stats['recall']:.4f}")
    # Accumulate for overall statistics
    recalls.append(stats['recall'])
    tot_missing_code.append(stats['num_missing_code'])
    tot_non_valid_predictions.append(stats['num_entries'] - stats['num_valid_predictions'])

print(f"\n")
print("=============================================================================================================")
overall_recall_mean = np.mean(recalls)
overall_recall_std = np.std(recalls)
print(f"Overall Recall across all programs: {overall_recall_mean:.4f} ± {overall_recall_std:.4f}")
total_missing_code = sum(tot_missing_code)
total_non_valid_predictions = sum(tot_non_valid_predictions)
print(f"Total missing code entries across all programs: {total_missing_code}")
print(f"Total non-valid prediction entries across all programs: {total_non_valid_predictions}")
print(f"Total entries analyzed: {df.shape[0]}")

-------------------------------------------------------------------------------------------------------------
Program: CParserXML
  Number of entries: 362
  Number of valid predictions: 361
  Number of missing code: 0
  Predicted label distribution: 0s=158, 1s=203, unknown=1
  True Positives: 84
  Actual Positives: 203
  Recall: 0.4138
-------------------------------------------------------------------------------------------------------------
Program: PicoHTTPParser
  Number of entries: 760
  Number of valid predictions: 734
  Number of missing code: 0
  Predicted label distribution: 0s=526, 1s=208, unknown=26
  True Positives: 19
  Actual Positives: 320
  Recall: 0.0594
-------------------------------------------------------------------------------------------------------------
Program: CSimpleJSONParser
  Number of entries: 653
  Number of valid predictions: 653
  Number of missing code: 0
  Predicted label distribution: 0s=393, 1s=260, unknown=0
  True Positives: 20
  Actual Positi

### Assembly Code LLM Classifications analysis

In [4]:
df = pd.read_csv("../Results/llm_classification_qwen3_coder_30b/global_predictions_assembly_code.csv")
# Shape of the dataframe
print(f"df.shape: {df.shape}")
# Display first few rows
df.head()

df.shape: (7898, 6)


,filename,function_name,address,assembly_code,predicted_label,true_label
0,parse_xml,sym.deregister_tm_clones,0x401110,"0x401110:\tmov\teax, 0x405058\n0x401115:\tcmp\...",0,0
1,parse_xml,sym.register_tm_clones,0x401140,"0x401140:\tmov\tesi, 0x405058\n0x401145:\tsub\...",0,0
2,parse_xml,sym.__do_global_dtors_aux,0x401180,0x401180:\tendbr64\t\n0x401184:\tcmp\tbyte ptr...,0,0
3,parse_xml,sym.frame_dummy,0x4011b0,0x4011b0:\tendbr64\t\n0x4011b4:\tjmp\t0x401140,0,0
4,parse_xml,sym.skip_whitespace,0x4011cb,"0x40120a:\tadd\tqword ptr [rbp - 8], 1\n0x4012...",0,1


In [5]:
# Remove the filename column and add: program, compiler, opt_level
df = enrich_program_metadata(df, drop_filename=True)

# Shape of the dataframe
print(f"df.shape: {df.shape}")
# Display first few rows
df.head()

df.shape: (7898, 8)


,function_name,address,assembly_code,predicted_label,true_label,program,opt_level,compiler
0,sym.deregister_tm_clones,0x401110,"0x401110:\tmov\teax, 0x405058\n0x401115:\tcmp\...",0,0,CParserXML,default,gcc
1,sym.register_tm_clones,0x401140,"0x401140:\tmov\tesi, 0x405058\n0x401145:\tsub\...",0,0,CParserXML,default,gcc
2,sym.__do_global_dtors_aux,0x401180,0x401180:\tendbr64\t\n0x401184:\tcmp\tbyte ptr...,0,0,CParserXML,default,gcc
3,sym.frame_dummy,0x4011b0,0x4011b0:\tendbr64\t\n0x4011b4:\tjmp\t0x401140,0,0,CParserXML,default,gcc
4,sym.skip_whitespace,0x4011cb,"0x40120a:\tadd\tqword ptr [rbp - 8], 1\n0x4012...",0,1,CParserXML,default,gcc


In [6]:
# List of unique programs
programs = df["program"].unique()
programs

array(['CParserXML', 'PicoHTTPParser', 'CSimpleJSONParser', 'cJSON',
       'Benoitc_HTTP', 'YACC_Calculator', 'Network_Packet_Analyzer',
       'ELF_Parser', 'PCAP_Parser', 'Packcc'], dtype=object)

In [7]:
recalls = []                    # Store recall statistics for each program
tot_missing_code = []           # Store total missing code statistics
tot_non_valid_predictions = []  # Store total non-valid predictions statistics

# Compute and print recall statistics for each program
for prog in programs:
    stats = compute_program_recall(df, prog, code_column="assembly_code")
    print("-------------------------------------------------------------------------------------------------------------")
    print(f"Program: {prog}")
    print(f"  Number of entries: {stats['num_entries']}")
    print(f"  Number of valid predictions: {stats['num_valid_predictions']}")
    print(f"  Number of missing code: {stats['num_missing_code']}")
    print(f"  Predicted label distribution: 0s={stats['num_pred_0']}, 1s={stats['num_pred_1']}, unknown={stats['num_pred_unknown']}")
    print(f"  True Positives: {stats['true_positives']}")
    print(f"  Actual Positives: {stats['actual_positives']}")
    print(f"  Recall: {stats['recall']:.4f}")
    # Accumulate for overall statistics
    recalls.append(stats['recall'])
    tot_missing_code.append(stats['num_missing_code'])
    tot_non_valid_predictions.append(stats['num_entries'] - stats['num_valid_predictions'])

print(f"\n")
print("=============================================================================================================")
overall_recall_mean = np.mean(recalls)
overall_recall_std = np.std(recalls)
print(f"Overall Recall across all programs: {overall_recall_mean:.4f} ± {overall_recall_std:.4f}")
total_missing_code = sum(tot_missing_code)
total_non_valid_predictions = sum(tot_non_valid_predictions)
print(f"Total missing code entries across all programs: {total_missing_code}")
print(f"Total non-valid prediction entries across all programs: {total_non_valid_predictions}")
print(f"Total entries analyzed: {df.shape[0]}")

-------------------------------------------------------------------------------------------------------------
Program: CParserXML
  Number of entries: 362
  Number of valid predictions: 356
  Number of missing code: 0
  Predicted label distribution: 0s=236, 1s=120, unknown=6
  True Positives: 104
  Actual Positives: 198
  Recall: 0.5253
-------------------------------------------------------------------------------------------------------------
Program: PicoHTTPParser
  Number of entries: 760
  Number of valid predictions: 760
  Number of missing code: 0
  Predicted label distribution: 0s=299, 1s=461, unknown=0
  True Positives: 203
  Actual Positives: 334
  Recall: 0.6078
-------------------------------------------------------------------------------------------------------------
Program: CSimpleJSONParser
  Number of entries: 653
  Number of valid predictions: 653
  Number of missing code: 0
  Predicted label distribution: 0s=243, 1s=410, unknown=0
  True Positives: 157
  Actual Posi